# Lab 0 — Set up MongoDB Atlas + ingest a retrieval dataset

This is a one-time setup notebook. You'll:

1. Confirm your Atlas + Voyage credentials.
2. Pick a public **BEIR** retrieval benchmark dataset.
3. Split each document into chunks and embed the chunks with
   **`voyage-context-3`** — Voyage's contextualized chunk-embedding
   model — via the MongoDB-hosted endpoint.
4. Insert the chunks into a MongoDB Atlas collection.
5. Create a **vector search index** and a **lexical (BM25) search
   index** so that later notebooks can compare retrieval
   strategies on identical data.

The next three notebooks (Labs 1 / 2 / 3) all read from the
collection this notebook builds. You only need to run this lab
once per dataset.

## Step 1 — Prerequisites

Create a `.env` file at the repo root with these values:

```
VOYAGE_API_KEY=al-...        # Atlas → AI Models → API Keys
MONGODB_URI=mongodb+srv://...
OPENAI_API_KEY=sk-...        # only needed in Lab 3
```

Then install the Python packages:

```
pip3 install --break-system-packages \
    pymongo voyageai beir python-dotenv requests \
    openai pandas matplotlib nbformat
```

In [ ]:
import os, sys
# Notebooks live at the repo root; library modules live in src/.
_REPO_ROOT = os.path.abspath(os.getcwd())
_SRC = os.path.join(_REPO_ROOT, "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

In [ ]:
# Load credentials and sanity-check they exist.
from dotenv import load_dotenv
load_dotenv(os.path.join(_REPO_ROOT, ".env"))

VOYAGE_API_KEY   = os.environ["VOYAGE_API_KEY"]
MONGODB_URI      = os.environ["MONGODB_URI"]
MONGODB_BASE_URL = "https://ai.mongodb.com/v1"

print("VOYAGE_API_KEY loaded:", VOYAGE_API_KEY[:8] + "…")
print("MONGODB_URI    loaded:", MONGODB_URI.split("@")[-1][:40] + "…")

## Step 2 — Connect to Atlas

Open a `pymongo` client and pick a database + collection name.
We'll namespace each BEIR dataset to its own collection so you
can ingest as many as you want without overwriting.

In [ ]:
import pymongo

DB_NAME    = "voyage_context_demo"
DATASET    = "scifact"                  # pick from the table below
COLL_NAME  = f"chunks_{DATASET.replace('-', '_')}"

client = pymongo.MongoClient(MONGODB_URI)
db     = client[DB_NAME]
coll   = db[COLL_NAME]

print(f"Database  : {DB_NAME}")
print(f"Collection: {COLL_NAME}")

## Step 3 — Pick a BEIR dataset

[BEIR](https://github.com/beir-cellar/beir) ships eight retrieval
benchmark datasets. Each one is a `(corpus, queries, qrels)`
triple with human-curated relevance judgements. Pick one to
ingest — `scifact` is small (5.2k docs) and clean, so start
there.

| name | description |
|---|---|
| `scifact`    | scientific claim verification (300 queries / 5.2k abstracts) — *recommended for first run* |
| `nfcorpus`   | medical literature retrieval (323 queries / 3.6k docs) |
| `fiqa`       | financial Q&A — opinionated long answers (648 queries / 57k docs) |
| `arguana`    | counter-argument retrieval (1.4k queries / 8.7k arguments) |
| `scidocs`    | scientific paper retrieval (1k queries / 25k docs) |
| `trec-covid` | COVID-19 research retrieval (50 queries / 171k docs) |
| `touche2020` | controversial-topic arguments (49 queries / 382k docs) |
| `quora`      | duplicate-question retrieval (10k queries / 523k docs) |

In [ ]:
# Load the BEIR dataset. This downloads the raw files to
# /tmp/beir_datasets/<DATASET>/ on first run.
from lib import load_beir_dataset
corpus, queries, qrels, info = load_beir_dataset(DATASET)

print(f"{DATASET}: {info['description']}")
print(f"  corpus  : {len(corpus):,} documents")
print(f"  queries : {len(queries):,} (test split)")
print(f"  qrels   : {len(qrels):,} queries have relevance judgements")

In [ ]:
# Look at one document, one query, and the qrels for that query.
sample_qid = next(iter(qrels))
sample_did = next(iter(qrels[sample_qid]))

print("Sample query   :", repr(queries[sample_qid]))
print()
print("Sample document:")
print("  doc_id :", sample_did)
print("  title  :", repr(corpus[sample_did]['title']))
print("  text   :", repr(corpus[sample_did]['text'][:200] + '…'))
print()
print(f"qrels[{sample_qid!r}] = {qrels[sample_qid]}")
print("  (each value is the relevance grade: 0=not relevant, 1+=relevant)")

## Step 4 — Sample documents to ingest

BEIR corpora can be large. For a fast first pass, pick a small
sample. We'll bias the sample to include every doc that's
marked relevant by at least one query, so the test queries
actually have something to retrieve.

In [ ]:
import random
random.seed(42)

CORPUS_SAMPLE = 500

# Collect all doc_ids that are marked relevant by any qrel,
# then top up with a random sample from the rest until we have
# CORPUS_SAMPLE documents.
corpus_keys = set(corpus.keys())
must_include = list({
    did
    for q_qrels in qrels.values()
    for did, score in q_qrels.items()
    if score > 0 and did in corpus_keys
})[:CORPUS_SAMPLE]

remaining   = [d for d in corpus.keys() if d not in set(must_include)]
random_top  = random.sample(remaining, max(0, CORPUS_SAMPLE - len(must_include)))
sample_ids  = must_include + random_top
print(f"Sampling {len(sample_ids)} docs "
      f"({len(must_include)} guaranteed-relevant + {len(random_top)} random)")

## Step 5 — Split each document into chunks

`voyage-context-3` is a **chunk** embedding model. It expects
each document to be split into smaller passages first, then
embeds each passage *in the context of the whole document* so
the resulting vectors carry the broader meaning.

We use a recursive character splitter — it tries paragraph
boundaries first, then lines, then sentences, then words.
It's about 30 lines of code, so we leave it in `lib.split_text`
and just call it here.

In [ ]:
from lib import split_text

CHUNK_SIZE    = 1000      # chars (~250 tokens)
CHUNK_OVERLAP = 150

records = []
for did in sample_ids:
    doc  = corpus[did]
    full = f"{doc['title']}\n\n{doc['text']}" if doc['title'] else doc['text']
    for i, chunk in enumerate(split_text(full, CHUNK_SIZE, CHUNK_OVERLAP)):
        records.append({
            "doc_id"   : did,
            "chunk_idx": i,
            "title"    : doc['title'],
            "text"     : chunk,
        })

print(f"{len(sample_ids)} docs → {len(records)} chunks")
print(f"first chunk record:")
print(f"  doc_id    : {records[0]['doc_id']}")
print(f"  chunk_idx : {records[0]['chunk_idx']}")
print(f"  text      : {records[0]['text'][:120]!r}…")

## Step 6 — Embed each chunk with `voyage-context-3`

The contextualized-embeddings endpoint is a separate API from
the standard `/v1/embeddings`: the request shape is

```json
{
  "model" : "voyage-context-3",
  "inputs": [
    ["full_doc_text", "chunk_1", "chunk_2", ...],
    ["full_doc_2",    "chunk_1", "chunk_2", ...]
  ]
}
```

— each inner list contains all chunks of one document plus the
full doc text as an "anchor" so the chunks share context.
That logic (batching, retries, splitting docs that exceed the
token cap) is ~100 lines, so it lives in `lib.embed_contextualized`.

In [ ]:
from collections import defaultdict
from lib import embed_contextualized

# Group chunks back by parent doc.
chunks_per_doc = defaultdict(list)
for r in records:
    chunks_per_doc[r['doc_id']].append(r)

doc_chunk_pairs = []
doc_order = list(chunks_per_doc.keys())
for did in doc_order:
    full = (f"{corpus[did]['title']}\n\n{corpus[did]['text']}"
            if corpus[did]['title'] else corpus[did]['text'])
    doc_chunk_pairs.append((full, [r['text'] for r in chunks_per_doc[did]]))

# POST /v1/contextualizedembeddings  →  one vector per chunk.
vectors = embed_contextualized(doc_chunk_pairs)
print(f"{len(vectors)} embeddings returned, dims={len(vectors[0])}")

# Attach vectors back to records in the same order they were built.
flat = 0
for did in doc_order:
    for r in chunks_per_doc[did]:
        r['embedding'] = vectors[flat]
        flat += 1

## Step 7 — Insert chunks into MongoDB

Drop the collection (in case we re-ran ingest) and insert in
batches. Each chunk becomes one document; the embedding is
stored alongside the chunk text.

In [ ]:
coll.drop()

BATCH = 500
for i in range(0, len(records), BATCH):
    coll.insert_many(records[i : i + BATCH])

print(f"Inserted {coll.estimated_document_count()} chunks into {DB_NAME}.{COLL_NAME}")
print("schema of one stored chunk:")
print(list(coll.find_one().keys()))

## Step 8 — Create the vector search index

Atlas needs a **search index** to support `$vectorSearch`. The
index definition tells Atlas which field is the vector, how
many dimensions it has, and which similarity metric to use.

We use cosine similarity here because that's what
`voyage-context-3` is trained on.

In [ ]:
from pymongo.operations import SearchIndexModel

VECTOR_INDEX_NAME = "voyage_vector_index"
DIMS = len(records[0]['embedding'])

vector_index = SearchIndexModel(
    name=VECTOR_INDEX_NAME,
    type="vectorSearch",
    definition={
        "fields": [{
            "type"         : "vector",
            "path"         : "embedding",
            "numDimensions": DIMS,
            "similarity"   : "cosine",
        }]
    },
)

existing = {idx['name'] for idx in coll.list_search_indexes()}
if VECTOR_INDEX_NAME not in existing:
    coll.create_search_index(vector_index)
    print(f"Created vector index '{VECTOR_INDEX_NAME}' ({DIMS} dims, cosine)")
else:
    print(f"Index '{VECTOR_INDEX_NAME}' already exists.")

## Step 9 — Create the lexical (BM25) search index

Atlas Search's standard text index supports BM25-style scoring.
We index the `text` and `title` fields with the standard Lucene
analyzer. This is the index `$search` will use in Lab 1.

In [ ]:
TEXT_INDEX_NAME = "voyage_text_index"

text_index = SearchIndexModel(
    name=TEXT_INDEX_NAME,
    type="search",
    definition={
        "mappings": {
            "dynamic": False,
            "fields": {
                "text" : {"type": "string", "analyzer": "lucene.standard"},
                "title": {"type": "string", "analyzer": "lucene.standard"},
            },
        }
    },
)

if TEXT_INDEX_NAME not in {idx['name'] for idx in coll.list_search_indexes()}:
    coll.create_search_index(text_index)
    print(f"Created text index '{TEXT_INDEX_NAME}'")
else:
    print(f"Index '{TEXT_INDEX_NAME}' already exists.")

## Step 10 — Wait for indexes to become queryable

Atlas builds search indexes asynchronously. Vector indexes
usually take 30–60s; text indexes can be slower. Poll until
both are `queryable: True`.

In [ ]:
import time

WAIT_SECONDS = 300        # bail out after 5 minutes
targets = {VECTOR_INDEX_NAME, TEXT_INDEX_NAME}

for _ in range(WAIT_SECONDS // 5):
    statuses = {idx['name']: idx.get('queryable', False)
                for idx in coll.list_search_indexes()
                if idx['name'] in targets}
    print("  ", statuses)
    if all(statuses.values()):
        print("Both indexes are queryable.")
        break
    time.sleep(5)
else:
    print("Timed out waiting. First query may be slow.")

## Done

You now have:

- `voyage_context_demo.chunks_scifact` (or whichever dataset
  you chose) with chunk documents that each include a
  contextualized embedding.
- A `voyage_vector_index` for `$vectorSearch`.
- A `voyage_text_index` for `$search` (BM25).

**Next:** open **`01_evaluate_blackbox.ipynb`** to see how IR
evaluation works — we'll treat lexical (BM25) retrieval as a
"black box" and measure its quality with the same metrics you'd
use to compare any retrieval system.